<a href="https://colab.research.google.com/github/PENNY7-11/rag_sortinghat/blob/main/RAG_01_%E6%89%93%E9%80%A0%E4%BD%A0%E7%9A%84_RAG_%E7%B3%BB%E7%B5%B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG 1｜用 EmbeddingGemma 2 與 FAISS 建立中文 RAG 知識庫

這份 Notebook 使用「大學生打工權益」第一版 **16 篇教學摘要**。每篇文件產生一個向量，不切塊；接著建立 FAISS 精確搜尋索引，下載給第二份 Colab 使用。

**操作方式：請由上往下，一格一格執行。每格先讀說明，再執行程式並觀察輸出。**

你會學到：原始文件 → 擷取 RAG 文字與 metadata → embedding → FAISS → 匯出知識庫。

本程式不呼叫 LLM，不使用 Groq，也不需要讀取 Colab 中的 `Groq` 密鑰。第二份 Colab 才會讀取該密鑰，透過 AIsuite 回答，並使用 Gradio 展示。

準備教材：`rag-data.zip`。內容是教學改寫摘要，並非官方原文或個案法律意見。

可在 Colab 的「執行階段 → 變更執行階段類型」選擇 GPU；沒有 GPU 也能用 CPU，但下載及運算需等待。模型首次下載需要網路。

## 1. 安裝套件

我們使用 Sentence Transformers 載入模型、NumPy 處理向量，以及 FAISS 搜尋向量。這格固定本教材採用的套件版本，方便記錄與重現。

先執行本格，再執行下格的 import。如果你已在本次執行階段載入其他版本的相關套件，請在安裝後重新啟動工作階段，再從下一格繼續；不需要重跑安裝格。

In [ ]:
%pip install -q "sentence-transformers==6.1.0" "transformers==5.20.1" "faiss-cpu==1.15.1"

## 2. 檢查環境與集中設定

`BATCH_SIZE` 可以調小以降低記憶體使用量。`EXPECTED_DOCUMENT_COUNT` 只用來提醒教材篇數；換自己的資料時可設為 `None`，不會強制只能有16篇。

以下會列出套件版本和 CPU／GPU。Colab 常見的 T4 不支援原生 bfloat16，本程式會改用 float32；不使用 EmbeddingGemma 2 不相容的 float16。

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import importlib.metadata as package_metadata
import hashlib
import json
import re
import tempfile
import zipfile

import torch
import faiss
from IPython.display import display
from sentence_transformers import SentenceTransformer
from huggingface_hub import HfApi

In [ ]:
MODEL_ID = "google/embeddinggemma-2"
BATCH_SIZE = 4
EXPECTED_DOCUMENT_COUNT = 17  # 換自己的資料的大小
EXPORT_ZIP_NAME = "Harry Potter vector_embedding.zip"

# text 本身已含標題；這裡不再從 metadata 添加標題。
DOCUMENT_TEMPLATE = "title: none | text: {text}"
QUERY_TEMPLATE = "task: search result | query: {question}"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = (
    torch.bfloat16
    if DEVICE == "cuda" and torch.cuda.is_bf16_supported()
    else torch.float32
)
PACKAGE_NAMES = ["torch", "numpy", "sentence-transformers", "transformers", "faiss-cpu", "huggingface-hub"]
PACKAGE_VERSIONS = {name: package_metadata.version(name) for name in PACKAGE_NAMES}
print("裝置：", DEVICE, "／模型精度：", DTYPE)
if DEVICE == "cuda":
    print("GPU：", torch.cuda.get_device_name(0))
display(pd.DataFrame(PACKAGE_VERSIONS.items(), columns=["套件", "版本"]))

裝置： cuda ／模型精度： torch.bfloat16
GPU： Tesla T4


,套件,版本
0,torch,2.11.0+cu130
1,numpy,2.1.3
2,sentence-transformers,6.1.0
3,transformers,5.20.1
4,faiss-cpu,1.15.1
5,huggingface-hub,1.33.0


## 3. 上傳教材 ZIP

執行後選擇第一版教材 ZIP。程式只解開 TXT；README、CSV、問題集不會匯入知識庫。中文檔名由 Python 的 ZIP 工具讀取。

每次執行這格會建立新的暫存資料夾，避免上一次的教材混進來。若上傳失敗，重新執行本格即可。後面的格子請重新依序執行。

In [ ]:
from google.colab import files

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if len(zip_names) != 1:
    raise ValueError("請一次上傳一個教材 ZIP 檔。")

archive_name = zip_names[0]
archive_bytes = uploaded[archive_name]
SOURCE_ARCHIVE_SHA256 = hashlib.sha256(archive_bytes).hexdigest()
DATA_DIR = Path(tempfile.mkdtemp(prefix="rag_documents_"))

# 僅解開 TXT，並確保 ZIP 中的路徑不會寫到暫存資料夾外。
import io
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    members = [
        member for member in archive.infolist()
        if not member.is_dir()
        and member.filename.lower().endswith(".txt")
        and not any(part.startswith((".", "__MACOSX")) for part in Path(member.filename).parts)
    ]
    if not members:
        raise ValueError("ZIP 中找不到 TXT；請確認使用第一版教材，或準備自己的 TXT。")
    destinations = set()
    for member in members:
        target = (DATA_DIR / member.filename).resolve()
        if not target.is_relative_to(DATA_DIR.resolve()):
            raise ValueError(f"ZIP 包含不適當的檔案路徑：{member.filename}")
        if target in destinations:
            raise ValueError(f"ZIP 有重複檔名：{member.filename}")
        destinations.add(target)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(member))

txt_paths = sorted(DATA_DIR.rglob("*.txt"), key=lambda p: p.relative_to(DATA_DIR).as_posix())
# 亦支援大寫 .TXT。
txt_paths = sorted(
    [p for p in DATA_DIR.rglob("*") if p.is_file() and p.suffix.lower() == ".txt"],
    key=lambda p: p.relative_to(DATA_DIR).as_posix(),
)
print("教材：", archive_name)
print("TXT 數量：", len(txt_paths))
for path in txt_paths:
    print("•", path.relative_to(DATA_DIR).as_posix())
if EXPECTED_DOCUMENT_COUNT is not None and len(txt_paths) != EXPECTED_DOCUMENT_COUNT:
    print(f"提醒：本次為 {len(txt_paths)} 篇，與原教材的 {EXPECTED_DOCUMENT_COUNT} 篇不同；仍可繼續。")

Saving Harry Potter rag-data.zip to Harry Potter rag-data.zip
教材： Harry Potter rag-data.zip
TXT 數量： 17
• Harry Potter/01.基礎設定.txt
• Harry Potter/02.地理.txt
• Harry Potter/03.動物和植物.txt
• Harry Potter/04.血統.txt
• Harry Potter/05.政府與政治.txt
• Harry Potter/06.教育.txt
• Harry Potter/07.貨幣.txt
• Harry Potter/08.體育與遊戲.txt
• Harry Potter/09.通訊.txt
• Harry Potter/10.交通.txt
• Harry Potter/11.巫師媒體.txt
• Harry Potter/12.飲食.txt
• Harry Potter/13.魔法物品.txt
• Harry Potter/14.黑魔法物品.txt
• Harry Potter/15.咒語大全.txt
• Harry Potter/16.魔法畫像與幽靈.txt
• Harry Potter/17.其他補充.txt


## 4. 先看一篇原始文件

請觀察「標題」「來源網址」「教材整理日期」「【內容】」「【檢索練習問題】」等欄位。不是每個欄位都適合送進 embedding。

本教材只取標題與正文。若把文件裡的練習問題也送進去，搜尋可能直接命中題目，無法公平觀察正文的檢索效果。

可修改 `SAMPLE_INDEX`，換看其他文章。

In [ ]:
SAMPLE_INDEX = 0
sample_path = txt_paths[SAMPLE_INDEX]
try:
    sample_raw = sample_path.read_text(encoding="utf-8-sig")
except UnicodeDecodeError as exc:
    raise ValueError(f"{sample_path.name} 不是可讀取的 UTF-8 文字。") from exc
print("檔名：", sample_path.name)
print("=" * 50)
print(sample_raw)

檔名： 01.基礎設定.txt
一、 核心規則與魔法本質

《國際巫師聯合會保密法》（International Statute of Wizardry，1692年）：魔法世界與麻瓜（Muggle，無魔法能力的一般人）世界徹底隔絕的分水嶺。巫師必須嚴格隱藏魔法存在，違者將受魔法法律執行司嚴懲。
《對未成年巫師加以合理限制法案》（Decree for the Reasonable Restriction of Underage Sorcery）：規定未滿 17 歲的巫師在校外禁止使用魔法，魔法部能透過魔杖上的蹤絲（Trace）偵測未成年人施咒的地點。
核心禁忌：
三大不可饒恕咒（The Unforgivable Curses）：使用任何一個都會被判處阿茲卡班終身監禁。
奪魂咒（Imperius Curse）：完全控制受害者意志。
鑽心咒（Cruciatus Curse）：施加極端肉體痛苦。
索命咒（Killing Curse, Avada Kedavra）：無法被抵擋、瞬間奪走生命的致命咒語。
時間倒流限制：透過時間轉換器（Time-Turner）回到過去會面臨極大風險，干擾時間線可能導致無法挽回的災難（如歷史遭扭曲或自身消亡）。

二、 魔法社會與文化認知

麻瓜與啞炮（Squibs）：
麻瓜（Muggle）：完全沒有魔法血統與能力的普通人。
啞炮（Squib）：出生於純血或混血巫師家庭，卻天生不具備魔法能力的人（例如米里森·伯斯德的親戚或阿格斯·費爾奇）。雖然能看見霍格沃茨，但在魔法社會常處於邊緣。
默默然與默然者（Obscurial & Obscurus）：當幼小的巫師被迫壓抑、隱藏自己的魔法天賦時，其體內會產生一股黑暗且具破壞力的寄生魔法能量「默默然」。宿主（默然者）通常活不過十歲。
靈魂與死後世界：魔法界證實靈魂真實存在。死後可以化為幽靈留在世間（但無法真正感受溫暖或食物），或透過冥想盆（Pensieve）提取記憶。黑魔法中甚至能透過魂器（Horcrux）將靈魂分裂並藏匿以求永生。

三、 歷史時間軸錨點

霍格沃茨創立（約公元 10 世紀）：由戈德里克·格蘭芬多、薩拉查·斯萊特林、羅伊納·拉文克勞與赫爾加·赫奇帕奇四位偉大巫師共同創立。
鄧布利多與伏地魔崛起（20 世紀中葉至末期）：
1945年：阿不思·鄧布利多擊敗黑巫師蓋勒特·格林德沃。
1970年代-

## 5. 第一個可修改函式：取出 RAG 文字

`extract_rag_text(raw_text)` 回傳「標題＋【內容】正文」。找到【內容】後，讀到下一個【區段標題】為止；練習問題和提醒不會混入。

**換自己的資料，主要修改這個函式。** 比如你的文件以 `正文：` 開始，就修改擷取規則。若找不到正文，本函式明確報錯，不擅自把整份文件拿去 embedding。

這個回傳值會原樣保存；未來檢索後也將它交給 LLM，確保搜尋與回答使用同一份內容。

In [ ]:
def extract_rag_text(raw_text):
    """從本教材 TXT 取出標題與正文；換資料時修改這裡。

    相容 Python 3.11+ 新版正規表示式規則。
    """
    # 1. 統一換行符號並移除 BOM
    normalized = raw_text.lstrip("\ufeff").replace("\r\n", "\n").replace("\r", "\n")

    # 2. 精準切分：利用換行符號直接切出第一行
    # split('\n', 1) 代表只切第一次遇到的換行，效率高且百分之百不會有語法錯誤
    parts = normalized.split("\n", 1)

    title = parts[0].strip() if len(parts) > 0 else ""
    body = parts[1].strip() if len(parts) > 1 else ""

    # 如果整份文件切開後發現根本沒有第二行，或者第一行就是全部內容
    if not body and title:
        body = title
        title = ""

    # 3. 確保正文不是空的防呆機制
    if not body:
        raise ValueError(
            "找不到非空的正文內容；請檢查文字檔是否為空，或修改 extract_rag_text()。"
        )

    # 4. 依照原系統設計需求，以兩個換行符號拼接標題與正文回傳
    return f"{title}\n\n{body}" if title else body


# 測試執行
sample_rag_text = extract_rag_text(sample_raw)
print(sample_rag_text)
print("\n擷取後字元數：", len(sample_rag_text))

一、 核心規則與魔法本質

《國際巫師聯合會保密法》（International Statute of Wizardry，1692年）：魔法世界與麻瓜（Muggle，無魔法能力的一般人）世界徹底隔絕的分水嶺。巫師必須嚴格隱藏魔法存在，違者將受魔法法律執行司嚴懲。
《對未成年巫師加以合理限制法案》（Decree for the Reasonable Restriction of Underage Sorcery）：規定未滿 17 歲的巫師在校外禁止使用魔法，魔法部能透過魔杖上的蹤絲（Trace）偵測未成年人施咒的地點。
核心禁忌：
三大不可饒恕咒（The Unforgivable Curses）：使用任何一個都會被判處阿茲卡班終身監禁。
奪魂咒（Imperius Curse）：完全控制受害者意志。
鑽心咒（Cruciatus Curse）：施加極端肉體痛苦。
索命咒（Killing Curse, Avada Kedavra）：無法被抵擋、瞬間奪走生命的致命咒語。
時間倒流限制：透過時間轉換器（Time-Turner）回到過去會面臨極大風險，干擾時間線可能導致無法挽回的災難（如歷史遭扭曲或自身消亡）。

二、 魔法社會與文化認知

麻瓜與啞炮（Squibs）：
麻瓜（Muggle）：完全沒有魔法血統與能力的普通人。
啞炮（Squib）：出生於純血或混血巫師家庭，卻天生不具備魔法能力的人（例如米里森·伯斯德的親戚或阿格斯·費爾奇）。雖然能看見霍格沃茨，但在魔法社會常處於邊緣。
默默然與默然者（Obscurial & Obscurus）：當幼小的巫師被迫壓抑、隱藏自己的魔法天賦時，其體內會產生一股黑暗且具破壞力的寄生魔法能量「默默然」。宿主（默然者）通常活不過十歲。
靈魂與死後世界：魔法界證實靈魂真實存在。死後可以化為幽靈留在世間（但無法真正感受溫暖或食物），或透過冥想盆（Pensieve）提取記憶。黑魔法中甚至能透過魂器（Horcrux）將靈魂分裂並藏匿以求永生。

三、 歷史時間軸錨點

霍格沃茨創立（約公元 10 世紀）：由戈德里克·格蘭芬多、薩拉查·斯萊特林、羅伊納·拉文克勞與赫爾加·赫奇帕奇四位偉大巫師共同創立。
鄧布利多與伏地魔崛起（20 世紀中葉至末期）：
1945年：阿不思·鄧布利多擊敗黑巫師蓋勒特·格林德沃。
1970年代-1981年：伏地魔第一次崛起，爆

## 6. 第二個可修改函式：取出 metadata

`extract_metadata(raw_text, filename)` 回傳標題、網址、日期、檔名和文件性質。metadata 是追溯資料用的資訊，不會整包送進 embedding。

**換自己的資料，可在這裡增減欄位。** 例如食譜可以增加料理類型，課程介紹可以增加學期。來源與日期找不到時用 `None`，不自行捏造；標題缺失時使用檔名。

這裡的 `date` 是教材整理日期，不是法規生效日期。保留「教學改寫摘要」的性質，避免後續展示誤稱官方原文。

In [ ]:
def extract_metadata(raw_text, filename):
    """取出本教材的描述資訊；換資料時修改這裡。

    專為哈利波特設定集優化：自動以第一行作為標題，其餘欄位保留原系統預設值邏輯。
    """
    # 1. 統一換行符號並移除 BOM
    normalized = raw_text.lstrip("\ufeff").replace("\r\n", "\n").replace("\r", "\n")

    # 2. 自動切出第一行作為標題內容 (解決原版只抓到檔名數字的問題)
    parts = normalized.split("\n", 1)
    first_line = parts[0].strip() if len(parts) > 0 else ""

    # 3. 定義內建欄位提取器
    def read_field(name):
        match = re.search(
            rf"^{re.escape(name)}[：:][ \t]*(.*)$",
            normalized,
            flags=re.MULTILINE,
        )
        return match.group(1).strip() or None if match else None

    # 4. 動態組合中介資料（完全依循原程式碼的預設邏輯）
    return {
        "title": read_field("標題") or first_line or Path(filename).stem,
        "source_url": read_field("來源網址"),
        "date": read_field("教材整理日期"),
        "filename": filename,
        "document_type": read_field("資料類型") or "未標示",
    }


# 測試執行並以標準 JSON 格式輸出
sample_metadata = extract_metadata(
    sample_raw, sample_path.relative_to(DATA_DIR).as_posix()
)
print(json.dumps(sample_metadata, ensure_ascii=False, indent=2))


{
  "title": "一、 核心規則與魔法本質",
  "source_url": null,
  "date": null,
  "filename": "Harry Potter/01.基礎設定.txt",
  "document_type": "未標示"
}


## 7. 批次整理成統一格式

把兩個函式組合起來，每篇得到 `doc_id`、`text` 和 `metadata`。檔案先排序，再按順序給編號；只要教材及排序不變，編號便相同。新增文件可能改變編號，因此需要整套重建索引與對照。

請檢查列表中每篇是否有標題、正文及來源。缺少 metadata 會提醒；正文缺失則停止並指出檔名，讓你回到前兩格修改函式。

In [ ]:
# 1. 重新定義支援自動切塊的文本擷取函式 (將其融入迴圈前)
def extract_rag_text_list(raw_text):
    """將原始文本標準化，並自動以「一、」、「二、」、「三、」為依據切分成多個符合限制的小文件段落。"""
    normalized = raw_text.lstrip("\ufeff").replace("\r\n", "\n").replace("\r", "\n")
    parts = normalized.split("\n", 1)
    main_title = parts[0].strip() if len(parts) > 0 else ""
    content = parts[1].strip() if len(parts) > 1 else ""
    if not content:
        return [main_title]

    # 用正則表達式識別「一、」或「1.」等段落起點來進行切塊
    split_pattern = r"(?=\n(?:[一二三四五六七八九十百]+、|\d+[\.\s]+))"
    sections = re.split(split_pattern, "\n" + content)

    chunks = []
    for section in sections:
        section = section.strip()
        if not section:
            continue
        chunk_text = f"{main_title}\n\n{section}" if main_title else section
        chunks.append(chunk_text)
    return chunks

# 2. 批次處理所有檔案並進行動態切塊載入
documents = []
for number, path in enumerate(txt_paths, start=1):
    filename = path.relative_to(DATA_DIR).as_posix()
    try:
        raw_text = path.read_text(encoding="utf-8-sig")
        if "\ufffd" in raw_text:
            raise ValueError("文字含有替代字元，請檢查原始編碼。")

        # 呼叫切塊函式，這時同一個檔案會得到一個包含多個段落的串列 (List)
        rag_text_chunks = extract_rag_text_list(raw_text)
        metadata = extract_metadata(raw_text, filename)

    except (ValueError, UnicodeDecodeError) as exc:
        raise ValueError(f"處理失敗：{filename}；原因：{exc}") from exc

    # 迴圈將切碎後的子文件逐一打包，並給予獨立的子 ID（如 doc_015_0, doc_015_1）
    for sub_idx, chunk in enumerate(rag_text_chunks):
        sub_id = f"doc_{number:03d}_{sub_idx}"

        # 為了讓每個切塊的 Metadata 更精準，我們把 Metadata 中的 title 替換成該切塊的前兩行作為具體標題
        chunk_lines = chunk.split("\n")
        chunk_title = f"{chunk_lines[0]} - {chunk_lines[2]}" if len(chunk_lines) > 2 else chunk_lines[0]

        chunk_metadata = metadata.copy()
        chunk_metadata["title"] = chunk_title

        documents.append({
            "doc_id": sub_id,
            "text": chunk,
            "metadata": chunk_metadata
        })

if not documents:
    raise ValueError("沒有可建立索引的文件。")

# 建立後續檢查長度用的 index_doc_ids 變數
index_doc_ids = [doc["doc_id"] for doc in documents]
if len(set(index_doc_ids)) != len(index_doc_ids):
    raise ValueError("doc_id 重複，請檢查編號規則。")

# 產出切塊後的摘要表格
summary = pd.DataFrame([
    {"doc_id": doc["doc_id"], "標題": doc["metadata"]["title"],
     "正文含標題字元數": len(doc["text"]), "來源": doc["metadata"]["source_url"],
     "整理日期": doc["metadata"]["date"]}
    for doc in documents
])
display(summary)

for doc in documents:
    missing = [key for key in ("source_url", "date") if not doc["metadata"].get(key)]
    if missing:
        print("提醒：", doc["doc_id"], "缺少", ", ".join(missing))

print("第一筆統一格式：")
print(json.dumps(documents[0], ensure_ascii=False, indent=2))


,doc_id,標題,正文含標題字元數,來源,整理日期
0,doc_001_0,一、 核心規則與魔法本質 - 《國際巫師聯合會保密法》（International Stat...,506,None,None
1,doc_001_1,一、 核心規則與魔法本質 - 二、 魔法社會與文化認知,349,None,None
2,doc_001_2,一、 核心規則與魔法本質 - 三、 歷史時間軸錨點,275,None,None
3,doc_002_0,一、 霍格沃茨魔法學校與周邊 - 霍格沃茨城堡（Hogwarts Castle）：位於蘇格蘭...,352,None,None
4,doc_002_1,一、 霍格沃茨魔法學校與周邊 - 二、 倫敦及英國魔法核心,610,None,None
5,doc_002_2,一、 霍格沃茨魔法學校與周邊 - 三、 住宅與關押地點,328,None,None
6,doc_003_0,一、 神奇動物與危險生物（Beasts） - 鷹頭馬身有翼獸（Hippogriff）：擁有巨...,531,None,None
7,doc_003_1,一、 神奇動物與危險生物（Beasts） - 二、 魔法植物與草藥（Plants）,410,None,None
8,doc_003_2,一、 神奇動物與危險生物（Beasts） - 三、魔法部神奇動物管理控制司分類級別,138,None,None
9,doc_004_0,一、 巫師血統分類 - 純血（Pure-blood）：家族中完全沒有麻瓜血統，世代皆為巫師的...,341,None,None


提醒： doc_001_0 缺少 source_url, date
提醒： doc_001_1 缺少 source_url, date
提醒： doc_001_2 缺少 source_url, date
提醒： doc_002_0 缺少 source_url, date
提醒： doc_002_1 缺少 source_url, date
提醒： doc_002_2 缺少 source_url, date
提醒： doc_003_0 缺少 source_url, date
提醒： doc_003_1 缺少 source_url, date
提醒： doc_003_2 缺少 source_url, date
提醒： doc_004_0 缺少 source_url, date
提醒： doc_004_1 缺少 source_url, date
提醒： doc_004_2 缺少 source_url, date
提醒： doc_005_0 缺少 source_url, date
提醒： doc_005_1 缺少 source_url, date
提醒： doc_005_2 缺少 source_url, date
提醒： doc_006_0 缺少 source_url, date
提醒： doc_006_1 缺少 source_url, date
提醒： doc_006_2 缺少 source_url, date
提醒： doc_007_0 缺少 source_url, date
提醒： doc_007_1 缺少 source_url, date
提醒： doc_008_0 缺少 source_url, date
提醒： doc_008_1 缺少 source_url, date
提醒： doc_009_0 缺少 source_url, date
提醒： doc_009_1 缺少 source_url, date
提醒： doc_010_0 缺少 source_url, date
提醒： doc_010_1 缺少 source_url, date
提醒： doc_011_0 缺少 source_url, date
提醒： doc_011_1 缺少 source_url, date
提醒： doc_012_0 缺少 source_url, date
提醒： doc_012_1 

## 8. 載入 EmbeddingGemma 2

先查詢模型目前的確切版本（commit），再載入該版本，將版本記錄到輸出設定，方便第二份 Colab 使用相同模型。只啟用文字部分，使用適合裝置的精度。

首次下載需要等待。本步使用公開模型，不讀取 Groq 或其他密鑰。若下載失敗，先確認網路；模型架構不受支援時，確認安裝格已完成，並重新啟動工作階段。程式不會默默改用別的模型。

In [ ]:
try:
    MODEL_REVISION = HfApi().model_info(MODEL_ID, token=False).sha
    model = SentenceTransformer(
        MODEL_ID,
        revision=MODEL_REVISION,
        token=False,
        device=DEVICE,
        config_kwargs={"vision_config": None, "audio_config": None},
        model_kwargs={"dtype": DTYPE, "attn_implementation": "eager"},
    )
except Exception as exc:
    raise RuntimeError(
        "EmbeddingGemma 2 載入失敗。請確認網路、套件版本，必要時重新啟動工作階段；"
        "若 GPU 記憶體不足，改用 CPU。未替換成其他模型。"
    ) from exc

print("模型：", MODEL_ID)
print("版本：", MODEL_REVISION)
print("裝置：", model.device)
print("模型精度：", model.dtype)
print("向量維度：", model.get_sentence_embedding_dimension())
print("模型單筆 token 上限：", model.max_seq_length)

Loading weights:   0%|          | 0/413 [00:00<?, ?it/s]

模型： google/embeddinggemma-2
版本： f9972ac22e696a105c482202646979e33e4b5a1e
裝置： cuda:0
模型精度： torch.bfloat16
向量維度： 768
模型單筆 token 上限： 1000000000000000019884624838656


/tmp/ipykernel_1397/465943176.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("向量維度：", model.get_sentence_embedding_dimension())


## 9. 檢查輸入長度，避免靜默截斷

本模型採用不同的任務前綴來區分文件和問題。文件本身已有標題，所以組成 `title: none | text: {text}`；問題則用 `task: search result | query: {question}`。

下格計算實際送入模型的文件文字（含前綴）長度。超過上限便停止，不會只取前半段。換成長文件時，可以在後續課程加入 chunking，或重新設計文件單位。

In [ ]:
def extract_rag_text_list(raw_text):
    """將原始文本標準化，並自動以「一、」、「二、」、「三、」為依據切分成多個符合限制的小文件段落。"""
    # 1. 統一換行符號並移除 BOM
    normalized = raw_text.lstrip("\ufeff").replace("\r\n", "\n").replace("\r", "\n")

    # 2. 擷取第一行作為主標題
    parts = normalized.split("\n", 1)
    main_title = parts[0].strip() if len(parts) > 0 else ""
    content = parts[1].strip() if len(parts) > 1 else ""

    # 如果沒有第二行，直接當成單一文本回傳
    if not content:
        return [main_title]

    # 3. 使用正規表示式，尋找以「一、」、「二、」、「三、」等開頭的段落起點
    # 支援「一、」、「二、」、「十、」或數字「1.」、「2.」等開頭
    split_pattern = r"(?=\n(?:[一二三四五六七八九十百]+、|\d+[\.\s]+))"
    sections = re.split(split_pattern, "\n" + content)

    chunks = []
    for section in sections:
        section = section.strip()
        if not section:
            continue

        # 4. 將「主標題」與「小段落內容」重新組合，讓每個切塊都帶有前後文資訊，提升 RAG 效果
        if main_title:
            chunk_text = f"{main_title}\n\n{section}"
        else:
            chunk_text = section

        chunks.append(chunk_text)

    return chunks


## 10. 產生並觀察文件向量

一篇文件對應矩陣中的一列，欄數是向量維度。`prompt=""` 表示前綴已自行加入，不讓模型額外再加一次。

向量正規化後長度約為1，內積便等於 cosine similarity。下格也將資料轉為 FAISS 使用的 float32 連續陣列，檢查空向量、非有限值及形狀。

In [ ]:
# 1. 強制在這裡重新對照最新的 documents 數量產生輸入文本（根治數量不符的問題）
document_inputs = [
    DOCUMENT_TEMPLATE.format(text=doc["text"]) for doc in documents
]

# 2. 開始執行模型轉向量 (呼叫 model.encode)
embeddings = model.encode(
    document_inputs,
    prompt="",
    batch_size=BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

# 3. 轉成標準的 float32 矩陣
embeddings = np.ascontiguousarray(embeddings, dtype=np.float32)

# 4. 安全檢查機制
if embeddings.ndim != 2 or embeddings.shape[0] != len(documents):
    raise ValueError(
        f"向量形狀與文件數不符。向量數：{embeddings.shape[0]}, 文件數：{len(documents)}"
    )

if not np.isfinite(embeddings).all():
    raise ValueError("向量含有 NaN 或無限值。")

if np.any(np.linalg.norm(embeddings, axis=1) <= 0):
    raise ValueError("出現零向量。")

# 5. 向量正規化與維度確認
faiss.normalize_L2(embeddings)
DIMENSION = int(embeddings.shape[1])

if DIMENSION != int(model.get_sentence_embedding_dimension()):
    raise ValueError("實際向量維度與模型宣告不符。")

# 6. 成功印出結果
print("矩陣形狀（文件數, 向量維度）：", embeddings.shape)
print("第一篇向量的前10個數字：", embeddings[0, :10])
print("前5篇向量長度：", np.linalg.norm(embeddings[:5], axis=1))


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

矩陣形狀（文件數, 向量維度）： (42, 768)
第一篇向量的前10個數字： [ 0.01530692  0.00189812 -0.0108551   0.00612887  0.01902693 -0.02158824
  0.06244735  0.0120138   0.03317515 -0.1473367 ]
前5篇向量長度： [1.         1.         1.         1.         0.99999994]


/tmp/ipykernel_1397/3487352197.py:35: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  if DIMENSION != int(model.get_sentence_embedding_dimension()):


## 11. 建立 FAISS 精確搜尋索引

`IndexFlatIP` 比較問題向量與所有文件向量的內積。因為雙方都正規化，排名就是 cosine similarity 的排名。

FAISS 回傳的是向量編號，**不是原文**。`index_doc_ids` 保存向量順序與文件編號的對應，再透過 `doc_id` 找回正文與 metadata。

In [ ]:
index = faiss.IndexFlatIP(DIMENSION)
index.add(embeddings)
documents_by_id = {doc["doc_id"]: doc for doc in documents}
assert index.ntotal == len(index_doc_ids) == len(documents)
print("索引類型：", type(index).__name__)
print("索引向量數：", index.ntotal)
print("索引維度：", index.d)
display(pd.DataFrame({"FAISS編號": range(len(index_doc_ids)), "doc_id": index_doc_ids}))

索引類型： IndexFlatIP
索引向量數： 42
索引維度： 768


,FAISS編號,doc_id
0,0,doc_001_0
1,1,doc_001_1
2,2,doc_001_2
3,3,doc_002_0
4,4,doc_002_1
5,5,doc_002_2
6,6,doc_003_0
7,7,doc_003_1
8,8,doc_003_2
9,9,doc_004_0


## 12. 定義問題向量化與搜尋函式

問題也要用同一個 embedding 模型向量化，但使用查詢專用前綴。`top_k` 表示取回幾篇文件；程式限制它不超過實際文件數。

搜尋函式只做檢索，還沒有呼叫 LLM。精確搜尋保證依目前向量找到分數最高者，不保證文件一定足以回答問題。

In [ ]:
def embed_question(question):
    if not isinstance(question, str) or not question.strip():
        raise ValueError("請輸入非空白問題。")
    query_input = QUERY_TEMPLATE.format(question=question.strip())
    if count_input_tokens(query_input) > TOKEN_LIMIT:
        raise ValueError("問題超過模型輸入長度上限，請縮短問題。")
    query_vector = model.encode(
        [query_input], prompt="", convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=False,
    )
    query_vector = np.ascontiguousarray(query_vector, dtype=np.float32)
    if query_vector.shape != (1, DIMENSION) or not np.isfinite(query_vector).all():
        raise ValueError("問題向量的維度或數值不正確。")
    if np.linalg.norm(query_vector) <= 0:
        raise ValueError("問題產生零向量。")
    faiss.normalize_L2(query_vector)
    return query_vector


def search_documents(question, top_k=3):
    if isinstance(top_k, bool) or not isinstance(top_k, (int, np.integer)) or top_k < 1:
        raise ValueError("top_k 必須是大於0的整數。")
    query_vector = embed_question(question)
    scores, positions = index.search(query_vector, min(int(top_k), index.ntotal))
    results = []
    for rank, (position, score) in enumerate(zip(positions[0], scores[0]), start=1):
        doc_id = index_doc_ids[int(position)]
        results.append({"rank": rank, "score": float(score), **documents_by_id[doc_id]})
    return results

## 13. 試查一次，閱讀真正取回的內容

修改 `QUESTION` 和 `TOP_K`，重新執行本格即可查詢。先看標題及分數，再讀完整正文，判斷是否有足夠依據。

可以比較 `top_k=1`、`3`、`5`。也可問「政大附近哪家飲料店時薪最高？」：即使教材沒有答案，FAISS 仍會傳回最相近的文章。**相似度不是正確率，也不是模型有答案的機率。**

In [ ]:
QUESTION = "去霍格莫德可以逛甚麼店？"
TOP_K = 3
results = search_documents(QUESTION, TOP_K)
display(pd.DataFrame([
    {"排名": r["rank"], "doc_id": r["doc_id"], "標題": r["metadata"]["title"],
     "cosine_similarity": r["score"], "來源": r["metadata"]["source_url"]}
    for r in results
]))
for result in results:
    print(f"\n[{result['doc_id']}] {result['metadata']['title']} ({result['score']:.4f})")
    print(result["text"])
    print("來源：", result["metadata"]["source_url"] or "未提供")

,排名,doc_id,標題,cosine_similarity,來源
0,1,doc_002_1,一、 霍格沃茨魔法學校與周邊 - 二、 倫敦及英國魔法核心,0.783131,None
1,2,doc_002_0,一、 霍格沃茨魔法學校與周邊 - 霍格沃茨城堡（Hogwarts Castle）：位於蘇格蘭...,0.778740,None
2,3,doc_012_2,一、 霍格沃茨禮堂經典美食 - 三、 飲品與酒吧,0.756445,None



[doc_002_1] 一、 霍格沃茨魔法學校與周邊 - 二、 倫敦及英國魔法核心 (0.7831)
一、 霍格沃茨魔法學校與周邊

二、 倫敦及英國魔法核心

對角巷（Diagon Alley）：隱藏在倫敦鬧市中的魔法商業街，入口通常在破釜酒吧（The Leaky Cauldron）後方。
古靈閣巫師銀行（Gringotts Wizarding Bank）：由妖精經營的地下金庫，防禦極其森嚴。
奧利凡德魔杖店（Ollivanders）：自公元前 382 年起營業的頂級魔杖製造商。
摩金夫人長袍專賣店（Madam Malkin's Robes for All Occasions）：購買霍格沃茨制服與日常長袍之處。
韋斯萊魔法戲法店（Weasleys' Wizard Wheezes）：由弗雷德與喬治開設的搞笑道具店。
翻倒巷（Knockturn Alley）：與對角巷相鄰的黑暗狹巷，專門販售黑魔法物品與神祕學材料，博金-博克商店（Borgin and Burkes）便坐落於此。
魔法部（Ministry of Magic）：位於倫敦地底深處的英國巫師政府核心。訪客通常透過紅色電話亭沖水進入（遊客入口）。內部包含神祕事務司（Department of Mysteries）、魔法法律執行司等部門。
九又四分之三站台（Platform 9¾）：位於倫敦國王十字車站（King's Cross Station），夾在第九與第十站台之間，是搭乘霍格沃茨特快列車（Hogwarts Express）前往學校的唯一起點。
來源： 未提供

[doc_002_0] 一、 霍格沃茨魔法學校與周邊 - 霍格沃茨城堡（Hogwarts Castle）：位於蘇格蘭高地的古老城堡，擁有數不清的移動樓梯、會變幻位置的房間、密室與隱蔽通道。城堡周邊有黑湖（Great Lake）與禁林（Forbidden Forest）。 (0.7787)
一、 霍格沃茨魔法學校與周邊

霍格沃茨城堡（Hogwarts Castle）：位於蘇格蘭高地的古老城堡，擁有數不清的移動樓梯、會變幻位置的房間、密室與隱蔽通道。城堡周邊有黑湖（Great Lake）與禁林（Forbidden Forest）。
霍格莫德村（Hogsmeade Village）：英國唯一的純巫師聚落，鄰近霍格沃茨。著名地標包含：
三把掃帚酒吧（Th

## 14. 保存索引、文件、編號對照與設定

第二份 Colab 需要下面四個檔案，不能只拿 FAISS 索引：

| 檔案 | 用途 |
|---|---|
| `index.faiss` | 文件向量與搜尋索引 |
| `documents.json` | 取回後提供給 LLM 的正文與來源資訊 |
| `index_doc_ids.json` | FAISS 編號對應的文件編號 |
| `config.json` | 模型確切版本、前綴、維度、正規化、套件版本及檢查資訊 |

不保存模型權重、API 金鑰或其他環境變數。第二份 Colab 載入相同模型版本，只重新計算問題向量。

In [ ]:
EXPORT_DIR = Path(tempfile.mkdtemp(prefix="rag_index_"))
faiss.write_index(index, str(EXPORT_DIR / "index.faiss"))

for filename, data in [("documents.json", documents), ("index_doc_ids.json", index_doc_ids)]:
    (EXPORT_DIR / filename).write_text(
        json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8"
    )

config = {
    "format_version": "1.0",
    "created_at": datetime.now(ZoneInfo("Asia/Taipei")).isoformat(timespec="seconds"),
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "model_config_kwargs": {"vision_config": None, "audio_config": None},
    "dimension": DIMENSION,
    "normalize_embeddings": True,
    "index_type": "IndexFlatIP",
    "metric": "cosine_similarity_via_normalized_inner_product",
    "document_template": DOCUMENT_TEMPLATE,
    "query_template": QUERY_TEMPLATE,
    "encode_prompt": "",
    "max_seq_length": TOKEN_LIMIT,
    "document_count": len(documents),
    "document_unit": "one_document_one_vector",
    "document_text_includes_title": True,
    "package_versions": PACKAGE_VERSIONS,
    "build_device": DEVICE,
    "build_dtype": str(DTYPE),
    "source_archive": archive_name,
    "source_archive_sha256": SOURCE_ARCHIVE_SHA256,
    "file_sha256": {
        name: hashlib.sha256((EXPORT_DIR / name).read_bytes()).hexdigest()
        for name in ["index.faiss", "documents.json", "index_doc_ids.json"]
    },
}
(EXPORT_DIR / "config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("已保存：")
for path in sorted(EXPORT_DIR.iterdir()):
    print(path.name, f"({path.stat().st_size:,} bytes)")
print("\n模型版本及前綴已寫入 config.json。")

已保存：
config.json (1,389 bytes)
documents.json (73,569 bytes)
index.faiss (129,069 bytes)
index_doc_ids.json (632 bytes)

模型版本及前綴已寫入 config.json。


## 15. 重新載入，驗證第二份 Colab 所需的資料

從剛才保存的檔案重新載入，檢查完整性、文件編號與向量對應，再使用**同一個問題向量**比對保存前後的排名與分數。此處不重算問題向量，以免不同運算裝置的細微浮點誤差干擾索引保存檢查。

也將 FAISS 結果與 NumPy 的全矩陣內積排序比較，確認精確搜尋的意義。遇到完全同分時，兩套工具的同分排列順序可以不同。

In [ ]:
loaded_config = json.loads((EXPORT_DIR / "config.json").read_text(encoding="utf-8"))
for name, expected_hash in loaded_config["file_sha256"].items():
    actual_hash = hashlib.sha256((EXPORT_DIR / name).read_bytes()).hexdigest()
    assert actual_hash == expected_hash, f"檔案完整性檢查失敗：{name}"

loaded_index = faiss.read_index(str(EXPORT_DIR / "index.faiss"))
loaded_documents = json.loads((EXPORT_DIR / "documents.json").read_text(encoding="utf-8"))
loaded_ids = json.loads((EXPORT_DIR / "index_doc_ids.json").read_text(encoding="utf-8"))
assert loaded_index.ntotal == len(loaded_documents) == len(loaded_ids) == loaded_config["document_count"]
assert loaded_index.d == loaded_config["dimension"]
assert len(set(loaded_ids)) == len(loaded_ids)
assert set(loaded_ids) == {doc["doc_id"] for doc in loaded_documents}
assert loaded_documents == documents and loaded_ids == index_doc_ids

verification_query = embed_question(QUESTION)
k = min(TOP_K, index.ntotal)
before_scores, before_positions = index.search(verification_query, k)
after_scores, after_positions = loaded_index.search(verification_query, k)
np.testing.assert_array_equal(before_positions, after_positions)
np.testing.assert_allclose(before_scores, after_scores, atol=1e-6, rtol=1e-6)

all_scores = (embeddings @ verification_query[0]).ravel()
expected_top_scores = np.sort(all_scores)[::-1][:k]
np.testing.assert_allclose(after_scores[0], expected_top_scores, atol=1e-5, rtol=1e-5)
np.testing.assert_allclose(after_scores[0], all_scores[after_positions[0]], atol=1e-5, rtol=1e-5)
print("通過：檔案完整性、文件對照、索引維度、保存前後一致，以及精確搜尋檢查。")

通過：檔案完整性、文件對照、索引維度、保存前後一致，以及精確搜尋檢查。


## 16. 打包並下載知識庫

ZIP 根目錄包含前述四個檔案。下載後，留給第二份 Colab 上傳使用；模型權重不在 ZIP 裡。

如果你更換文件、修改擷取函式或 embedding 設定，請重新執行受影響步驟並重新匯出，不要混用舊索引與新文件。

In [ ]:
EXPORT_ZIP = EXPORT_DIR.parent / f"{EXPORT_DIR.name}_download" / EXPORT_ZIP_NAME
EXPORT_ZIP.parent.mkdir(parents=True, exist_ok=True)
required_files = ["index.faiss", "documents.json", "index_doc_ids.json", "config.json"]
with zipfile.ZipFile(EXPORT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for name in required_files:
        archive.write(EXPORT_DIR / name, arcname=name)
with zipfile.ZipFile(EXPORT_ZIP) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(required_files)
print("下載檔名：", EXPORT_ZIP_NAME)
print("大小：", f"{EXPORT_ZIP.stat().st_size:,} bytes")
files.download(str(EXPORT_ZIP))

下載檔名： Harry Potter vector_embedding.zip
大小： 112,002 bytes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 17. 換成自己的資料：修改清單與課堂練習

1. 準備一篇一檔的 UTF-8 TXT，包成 ZIP；文件格式由你設計。
2. 修改 **`extract_rag_text()`**：哪些文字要搜尋，也要提供給 LLM？
3. 修改 **`extract_metadata()`**：哪些欄位用於來源、日期、分類或顯示？
4. 將 `EXPECTED_DOCUMENT_COUNT` 設為 `None` 或新資料的預期數量。
5. 先看一篇擷取結果，再批次整理。確認沒有把範例問題或答案集混入正文。
6. 從批次整理步驟重新建立向量及索引，並重新匯出。若新增 metadata 欄位，第二份 Colab 的顯示也要配合修改。

**觀察問題：** 同一題改用 `top_k=1`、`3`、`5`，有哪些文章被取回？最高分文章能回答問題嗎？教材沒有答案時，為什麼搜尋仍有結果？

本份只完成檢索，不會讓 LLM 產生答案。下一份才會讀取 Colab Secrets 名稱為 **`Groq`** 的密鑰，設定 `GROQ_API_KEY`，透過 AIsuite 呼叫 `groq:openai/gpt-oss-120b`，最後以 Gradio 展示。

### 參考資料

- [EmbeddingGemma 2 官方 Sentence Transformers 指引](https://ai.google.dev/gemma/docs/embeddinggemma/inference-embeddinggemma-with-sentence-transformers)：文字模式、精度與檢索前綴。
- [EmbeddingGemma 2 模型](https://huggingface.co/google/embeddinggemma-2)
- [FAISS 入門與索引說明](https://github.com/facebookresearch/faiss/wiki/Getting-started)

教材採用一篇一向量的設計。法規內容可能隨日期變動；來源網址可供追溯，這些摘要不能取代當時有效的法規與個案判斷。